# Modelo Final V3 — Iteración Completa
## Caso Práctico · Empresa de Telecomunicaciones · Prácticas Aplicadas 2026

---

Esta iteración amplía el proyecto en dos direcciones:

**A. Mejor modelo**: LightGBM y XGBoost frente a la Regresión Logística.
Los modelos de Gradient Boosting suelen superar a LR en problemas
tabulares sin necesidad de escalar variables.

**B. Análisis de calidad**: SHAP values (importancia rigurosa de variables),
ablación de la variable dominante, curva de ganancia, umbral óptimo
de negocio y Walk-Forward Validation para medir estabilidad temporal.


## 1. Librerías


In [ ]:
import warnings, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.linear_model       import LogisticRegression
from sklearn.preprocessing      import StandardScaler, OneHotEncoder
from sklearn.pipeline           import Pipeline
from sklearn.impute             import SimpleImputer
from sklearn.compose            import ColumnTransformer
from sklearn.metrics            import (roc_auc_score, average_precision_score,
                                        recall_score, precision_score, f1_score)
from lightgbm import LGBMClassifier
from xgboost  import XGBClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid', context='notebook')

# Colores del notebook (consistentes en todos los gráficos)
C_CHURN   = '#E85C4C'   # rojo   = churn / riesgo / malo
C_SAFE    = '#4C9BE8'   # azul   = seguro / no churn
C_BEST    = '#2DC653'   # verde  = mejor resultado

ROOT = Path('..').resolve()
sys.path.append(str(ROOT))
from src.load  import load_all
from src.clean import clean_all

RANDOM_STATE = 42
print('Librerías cargadas')


## 2. Carga de datos y construcción del panel

Mismo pipeline que en modelo_features_v2. Todas las variables usan **lag 1**:
para predecir si un cliente abandona en el mes T, sólo usamos datos de T-1.
Esto garantiza cero leakage.


In [ ]:
data  = load_all()
clean = clean_all(data, save=False)
clientes  = clean['clientes']
churn_raw = clean['churn']
factura   = clean['facturacion']
soporte   = clean['soporte']
calidad   = clean['calidad']
encuestas = clean['encuestas']
print('Datos cargados')


In [ ]:
# ── Facturación: lag 1 de variables de pago + rolling 3m + señales de tendencia
factura['fecha']   = pd.to_datetime(factura['fecha'],   format='mixed', dayfirst=True)
churn_raw['fecha'] = pd.to_datetime(churn_raw['fecha'], format='mixed', dayfirst=True)
factura['ym']   = factura['fecha'].dt.to_period('M')
churn_raw['ym'] = churn_raw['fecha'].dt.to_period('M')
factura = factura.sort_values(['cliente_id', 'ym'])

fac_lag = factura.assign(
    impago_mes_lag1    = lambda d: d.groupby('cliente_id')['impago_flag'].shift(1),
    stress_mes_lag1    = lambda d: d.groupby('cliente_id')['stress_calidad_lag'].shift(1),
    dias_retraso_lag1  = lambda d: d.groupby('cliente_id')['dias_retraso_pago'].shift(1),
    consumo_extra_lag1 = lambda d: d.groupby('cliente_id')['consumo_extra'].shift(1),
    variacion_lag1     = lambda d: d.groupby('cliente_id')['variacion_consumo_pct'].shift(1),
)[['cliente_id','ym','impago_mes_lag1','stress_mes_lag1',
   'dias_retraso_lag1','consumo_extra_lag1','variacion_lag1']]

# Media de los 3 meses anteriores para suavizar picos puntuales
roll3 = (
    factura.groupby('cliente_id')[
        ['impago_flag','stress_calidad_lag','dias_retraso_pago','variacion_consumo_pct']
    ]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    .rename(columns={
        'impago_flag':          'impago_roll3',
        'stress_calidad_lag':   'stress_roll3',
        'dias_retraso_pago':    'retraso_roll3',
        'variacion_consumo_pct':'variacion_roll3',
    })
)
fac_lag = pd.concat([fac_lag, roll3], axis=1)

# Cuántos meses consecutivos lleva sin pagar
fac_lag['racha_impagos_lag1'] = (
    factura.groupby('cliente_id')['impago_flag']
    .transform(lambda s: s.shift(1).groupby((s.shift(1)==0).cumsum()).cumcount())
)

# Flag: sin consumo extra los últimos 2 meses (señal de desenganche)
fac_lag['sin_consumo_2m_lag1'] = (
    factura.groupby('cliente_id')['consumo_extra']
    .transform(lambda s: (s.shift(1).rolling(2, min_periods=2).max()==0).astype(int))
)

# Ratio estrés multicausal: combina retraso de pago + mala red
# Valor alto = cliente que paga tarde Y tiene mala red = máximo riesgo
fac_lag['ratio_estres_lag1'] = (
    (fac_lag['dias_retraso_lag1'].fillna(0) + 1) /
    (fac_lag['stress_mes_lag1'].fillna(0.5)  + 1)
)

print(f'Features facturación: {fac_lag.shape}')


In [ ]:
# ── Soporte: interacciones e incidencias críticas del mes anterior ────
soporte['fecha_evento'] = pd.to_datetime(soporte['fecha_evento'], format='mixed', dayfirst=True)
soporte['ym'] = soporte['fecha_evento'].dt.to_period('M')

sop_mes = soporte.groupby(['cliente_id','ym']).agg(
    n_interacciones_mes = ('interaccion_id', 'count'),
    n_llamadas_baja_mes = ('motivo', lambda x: (x=='Baja/Portabilidad').sum()),
    # 1 si hubo alguna incidencia crítica ese mes (queja de facturación o intento de baja)
    critica_mes = ('motivo', lambda x: int(x.isin(['Facturación','Baja/Portabilidad']).any())),
).reset_index()

sop_lag = (
    sop_mes.sort_values(['cliente_id','ym'])
    .assign(
        interacc_lag1          = lambda d: d.groupby('cliente_id')['n_interacciones_mes'].shift(1),
        baja_lag1              = lambda d: d.groupby('cliente_id')['n_llamadas_baja_mes'].shift(1),
        critica_pendiente_lag1 = lambda d: d.groupby('cliente_id')['critica_mes'].shift(1),
    )
    [['cliente_id','ym','interacc_lag1','baja_lag1','critica_pendiente_lag1']]
)

# ── Calidad de red por zona: índice del mes anterior ─────────────────
calidad['fecha'] = pd.to_datetime(calidad['fecha'], format='mixed', dayfirst=True)
calidad['ym']    = calidad['fecha'].dt.to_period('M')
cal_zona = calidad.groupby(['zona_id','ym']).agg(
    calidad_global_lag1 = ('indice_calidad_global', 'mean'),
    tasa_cortes_lag1    = ('tasa_cortes_pct',        'mean'),
    cobertura_5g_lag1   = ('cobertura_5g_pct',        'mean'),
).reset_index()
cal_zona_lag = cal_zona.copy()
cal_zona_lag['ym'] = cal_zona_lag['ym'] + 1   # desplazar al mes siguiente

# ── Encuestas: sentimiento medio de la zona el mes anterior ──────────
encuestas['fecha'] = pd.to_datetime(encuestas['fecha'], format='mixed', dayfirst=True)
encuestas['ym']    = encuestas['fecha'].dt.to_period('M')
enc_zona = (
    encuestas.groupby(['zona_id','ym'])['sent_text_latente']
    .mean().reset_index(name='sentimiento_lag1')
)
enc_zona_lag = enc_zona.copy()
enc_zona_lag['ym'] = enc_zona_lag['ym'] + 1

print('Soporte, calidad y encuestas listos')


In [ ]:
# ── Unión de todas las fuentes en el panel cliente-mes ────────────────
panel = churn_raw.merge(
    clientes[['cliente_id','zona_id','region','tipo_zona','tipo_plan',
              'edad','sexo','estado_civil','num_lineas',
              'ingreso_estimado','antiguedad_meses','descuento_activo']],
    on='cliente_id', how='left'
)
panel = panel.merge(fac_lag,                                           on=['cliente_id','ym'], how='left')
panel = panel.merge(sop_lag,                                           on=['cliente_id','ym'], how='left')
panel = panel.merge(cal_zona_lag[['zona_id','ym','calidad_global_lag1',
                                   'tasa_cortes_lag1','cobertura_5g_lag1']], on=['zona_id','ym'], how='left')
panel = panel.merge(enc_zona_lag[['zona_id','ym','sentimiento_lag1']],  on=['zona_id','ym'], how='left')

# Eliminamos el primer mes de cada cliente (no tiene lag, no podemos predecir)
panel = panel.dropna(subset=['impago_mes_lag1'])
panel['churn'] = panel['churn'].astype(int)

# Codificación ordinal del plan: respeta el orden natural Básico < Contrato < Premium
panel['tipo_plan_enc'] = panel['tipo_plan'].map(
    {'Básico':1, 'Prepago':1, 'Contrato':2, 'Premium':3}
).fillna(2)

print(f'Panel final: {panel.shape[0]:,} filas x {panel.shape[1]} columnas')
print(f'Tasa de churn mensual: {panel.churn.mean():.4f}  ({panel.churn.mean():.2%})')
print(f'Desbalance: 1 churner por cada {int(1/panel.churn.mean())} no-churners')


## 3. Split train/test 80/20 por cliente

El split se hace **por cliente entero**: todos los meses de un cliente van
al mismo conjunto. Esto evita que el modelo vea meses de enero de un cliente
en train y sus meses de julio en test, lo que contaminaría los resultados.


In [ ]:
FEATURES_NUM = [
    'impago_mes_lag1', 'stress_mes_lag1', 'dias_retraso_lag1',
    'consumo_extra_lag1', 'variacion_lag1',
    'impago_roll3', 'stress_roll3', 'retraso_roll3', 'variacion_roll3',
    'racha_impagos_lag1', 'sin_consumo_2m_lag1',
    'ratio_estres_lag1', 'critica_pendiente_lag1', 'sentimiento_lag1',
    'interacc_lag1', 'baja_lag1',
    'calidad_global_lag1', 'tasa_cortes_lag1', 'cobertura_5g_lag1',
    'edad', 'ingreso_estimado', 'antiguedad_meses', 'num_lineas',
    'tipo_plan_enc',
]
FEATURES_CAT = ['tipo_zona', 'region']
TARGET = 'churn'

clientes_unicos = panel['cliente_id'].unique()
rng = np.random.default_rng(RANDOM_STATE)
rng.shuffle(clientes_unicos)
n_train   = int(len(clientes_unicos) * 0.8)
train_ids = set(clientes_unicos[:n_train])
test_ids  = set(clientes_unicos[n_train:])

train = panel[panel['cliente_id'].isin(train_ids)].copy()
test  = panel[panel['cliente_id'].isin(test_ids)].copy()

X_train = train[FEATURES_NUM + FEATURES_CAT]
y_train = train[TARGET]
X_test  = test[FEATURES_NUM + FEATURES_CAT]
y_test  = test[TARGET]

print(f'Train: {len(train):,} filas | {len(train_ids):,} clientes | tasa churn: {y_train.mean():.4f}')
print(f'Test:  {len(test):,} filas  | {len(test_ids):,} clientes  | tasa churn: {y_test.mean():.4f}')
print(f'Solapamiento de clientes entre conjuntos: {len(train_ids & test_ids)} (debe ser 0)')

# Desbalance para los modelos que lo necesitan (scale_pos_weight)
scale_pos = int((y_train==0).sum() / (y_train==1).sum())
print(f'scale_pos_weight para GBM: {scale_pos}')


In [ ]:
# Preprocesadores
# LR necesita escalado; GBM no, pero sí necesita manejo de nulos y categóricas

pre_num = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])
pre_cat = Pipeline([
    ('imp', SimpleImputer(strategy='constant', fill_value='desconocido')),
    ('enc', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False)),
])
prep_lr = ColumnTransformer([('num', pre_num, FEATURES_NUM), ('cat', pre_cat, FEATURES_CAT)])

# GBM: sin escalado pero con imputer y encoder
pre_num_gbm = Pipeline([('imp', SimpleImputer(strategy='median'))])
prep_gbm = ColumnTransformer([('num', pre_num_gbm, FEATURES_NUM), ('cat', pre_cat, FEATURES_CAT)])

print('Preprocesadores definidos')


## 4. Comparativa: Regresión Logística vs Gradient Boosting

Probamos tres modelos. El objetivo no es el AUC más alto posible sino
entender qué gana el Gradient Boosting frente a la LR que ya teníamos.

| Modelo | Por qué lo probamos |
|--------|-------------------|
| LR (referencia) | Mejor modelo de V2, AUC = 0.703. Punto de partida |
| LightGBM | GBM moderno, muy eficiente, estándar en competiciones de datos tabulares |
| XGBoost | Otra implementación de GBM, referencia del sector |


In [ ]:
def evaluar(nombre, y_real, y_prob, umbral=0.10):
    """Calcula las métricas principales para un modelo dado."""
    y_pred = (y_prob >= umbral).astype(int)
    return {
        'Modelo':    nombre,
        'AUC-ROC':   round(roc_auc_score(y_real, y_prob), 4),
        'PR-AUC':    round(average_precision_score(y_real, y_prob), 4),
        'Recall':    round(recall_score(y_real, y_pred, zero_division=0), 3),
        'Precisión': round(precision_score(y_real, y_pred, zero_division=0), 3),
        'F1':        round(f1_score(y_real, y_pred, zero_division=0), 3),
    }

resultados = []

# ── Regresión Logística ───────────────────────────────────────────────
lr_pipe = Pipeline([
    ('prep',  prep_lr),
    ('model', LogisticRegression(
        C=0.01, penalty='l1', solver='saga',
        class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE
    ))
])
lr_pipe.fit(X_train, y_train)
prob_lr = lr_pipe.predict_proba(X_test)[:, 1]
resultados.append(evaluar('LR (referencia V2)', y_test, prob_lr))
print(f'LR     AUC = {roc_auc_score(y_test, prob_lr):.4f}')

# ── LightGBM ──────────────────────────────────────────────────────────
lgbm_pipe = Pipeline([
    ('prep',  prep_gbm),
    ('model', LGBMClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=5,
        num_leaves=31, min_child_samples=50,
        scale_pos_weight=scale_pos,   # equivalente a class_weight='balanced'
        random_state=RANDOM_STATE, verbose=-1
    ))
])
lgbm_pipe.fit(X_train, y_train)
prob_lgbm = lgbm_pipe.predict_proba(X_test)[:, 1]
resultados.append(evaluar('LightGBM', y_test, prob_lgbm))
print(f'LGBM   AUC = {roc_auc_score(y_test, prob_lgbm):.4f}')

# ── XGBoost ───────────────────────────────────────────────────────────
xgb_pipe = Pipeline([
    ('prep',  prep_gbm),
    ('model', XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos,
        random_state=RANDOM_STATE, eval_metric='auc', verbosity=0
    ))
])
xgb_pipe.fit(X_train, y_train)
prob_xgb = xgb_pipe.predict_proba(X_test)[:, 1]
resultados.append(evaluar('XGBoost', y_test, prob_xgb))
print(f'XGBoost AUC = {roc_auc_score(y_test, prob_xgb):.4f}')

df_res = pd.DataFrame(resultados).set_index('Modelo')
print()
print(df_res.to_string())


In [ ]:
# El gráfico solo muestra AUC — una barra por modelo, un color para el mejor
aucs    = df_res['AUC-ROC'].tolist()
modelos = df_res.index.tolist()
max_auc = max(aucs)
colores = [C_BEST if a == max_auc else C_SAFE for a in aucs]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(modelos, aucs, color=colores, height=0.45, edgecolor='white')
for bar, val in zip(bars, aucs):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontweight='bold', fontsize=11)
ax.axvline(0.703, color=C_CHURN, linestyle='--', linewidth=1.5,
           label='Referencia V2 = 0.703')
ax.set_xlabel('AUC-ROC')
ax.set_title('Comparativa de modelos', fontweight='bold')
ax.set_xlim(0.65, 0.80)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

mejor_nombre = df_res['AUC-ROC'].idxmax()
mejor_auc    = df_res['AUC-ROC'].max()
prob_mejor   = {'LR (referencia V2)': prob_lr,
                'LightGBM': prob_lgbm,
                'XGBoost': prob_xgb}[mejor_nombre]
print(f'Mejor modelo: {mejor_nombre}  AUC = {mejor_auc:.4f}')


## 5. SHAP — Importancia rigurosa de variables

Los coeficientes de LR son difíciles de interpretar cuando las variables
están correladas. SHAP calcula, para cada predicción concreta, cuánto
contribuyó cada variable. Tiene garantías teóricas de ser justo y consistente.

Lo aplicamos sobre LightGBM (el modelo más potente).

Si la variable más importante concentra más del 40% de la importancia
total, hacemos **ablación**: la quitamos y reentrenamos para comprobar
si las demás variables pueden compensar su ausencia.


In [ ]:
# Transformar los datos de test con el preprocesador de LightGBM
X_test_prep = lgbm_pipe.named_steps['prep'].transform(X_test)

# Recuperar los nombres de las features después del OneHotEncoder
cat_names = (
    lgbm_pipe.named_steps['prep']
    .named_transformers_['cat']
    .named_steps['enc']
    .get_feature_names_out(FEATURES_CAT)
    .tolist()
)
feat_names = FEATURES_NUM + cat_names

# Calcular SHAP values (TreeExplainer es rápido para modelos de árbol)
explainer   = shap.TreeExplainer(lgbm_pipe.named_steps['model'])
shap_values = explainer.shap_values(X_test_prep)

# shap_values puede ser lista [clase_0, clase_1] según la versión
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

# Importancia = media del valor absoluto de SHAP por variable
imp_shap = (
    pd.Series(np.abs(sv).mean(axis=0), index=feat_names)
    .sort_values(ascending=False)
)

pct_top1 = imp_shap.iloc[0] / imp_shap.sum()
print(f'Variable más importante: {imp_shap.index[0]}')
print(f'  Porcentaje del total: {pct_top1:.1%}')
print()
print('Top 10 por SHAP:')
print(imp_shap.head(10).to_string())


In [ ]:
# Beeswarm SHAP: cada punto es una predicción del conjunto de test.
# Eje X: cuánto empuja esa variable hacia churn (derecha) o no-churn (izquierda).
# Color: valor de la variable (rojo=alto, azul=bajo).
muestra = min(2000, X_test_prep.shape[0])
idx_m   = np.random.default_rng(42).choice(X_test_prep.shape[0], muestra, replace=False)

# Valor base del modelo (predicción media sin ninguna variable)
base_val = (explainer.expected_value[1]
            if isinstance(explainer.expected_value, list)
            else explainer.expected_value)

shap_exp = shap.Explanation(
    values        = sv[idx_m],
    base_values   = base_val,
    data          = X_test_prep[idx_m],
    feature_names = feat_names
)

plt.figure(figsize=(10, 8))
shap.plots.beeswarm(shap_exp, max_display=15, show=False)
plt.title('SHAP — Impacto de cada variable en la probabilidad de churn',
          fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


In [ ]:
# ── Ablación: ¿qué pasa si quitamos la variable dominante? ───────────
# Si una variable tiene >40% de importancia, puede estar eclipsando a las demás.
# La quitamos para ver si el modelo sigue funcionando sin ella.

var_dom    = imp_shap.index[0]
pct_dom    = pct_top1

print(f'Variable dominante: {var_dom}  ({pct_dom:.1%} del total SHAP)')

if pct_dom > 0.40:
    print(f'Supera el 40% → realizamos ablación')

    feats_sin = [f for f in FEATURES_NUM if f != var_dom]
    prep_abl  = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), feats_sin),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='desconocido')),
            ('enc', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False)),
        ]), FEATURES_CAT),
    ])
    lgbm_abl = Pipeline([
        ('prep',  prep_abl),
        ('model', LGBMClassifier(
            n_estimators=500, learning_rate=0.05, max_depth=5,
            num_leaves=31, min_child_samples=50,
            scale_pos_weight=scale_pos, random_state=RANDOM_STATE, verbose=-1
        ))
    ])
    lgbm_abl.fit(train[feats_sin + FEATURES_CAT], y_train)
    prob_abl = lgbm_abl.predict_proba(test[feats_sin + FEATURES_CAT])[:, 1]
    auc_abl  = roc_auc_score(y_test, prob_abl)

    print(f'  AUC con todas las variables:   {mejor_auc:.4f}')
    print(f'  AUC sin {var_dom}: {auc_abl:.4f}')
    diff = auc_abl - mejor_auc
    print(f'  Diferencia: {diff:+.4f}')
    if diff > -0.005:
        print('  Conclusión: la pérdida es mínima.')
        print('  Las demás variables compensan bien. El modelo sin esta variable')
        print('  es más generalizable y menos dependiente de una sola señal.')
    else:
        print('  Conclusión: la variable aporta valor real. Conviene mantenerla.')
else:
    print(f'No supera el 40% ({pct_dom:.1%}) — la importancia está bien distribuida.')
    print('No realizamos ablación.')
    auc_abl = None


## 6. Umbral de decisión óptimo

Con la tasa de churn del 0.63%, el umbral por defecto de 0.5 no tiene ningún
sentido: el modelo nunca detectaría un churner. Hay que calibrar el umbral
según el coste de negocio de cada tipo de error:

- **Falso Negativo**: no detectamos a un churner → perdemos el cliente
- **Falso Positivo**: avisamos de un cliente que no iba a irse → desperdiciamos el coste de contacto

El umbral óptimo es el que maximiza el **beneficio neto** de la campaña.


In [ ]:
# Supuestos de negocio (ajustar con datos reales de la empresa)
INGRESO_MENSUAL  = 128    # € por cliente al mes
MESES_VIDA_EXTRA = 12     # meses adicionales conseguidos con retención exitosa
TASA_EXITO       = 0.30   # % de contactos que resultan en retención real
COSTE_CONTACTO   = 30     # € por llamada / contacto

valor_retencion = INGRESO_MENSUAL * MESES_VIDA_EXTRA * TASA_EXITO
print(f'Valor de retener un churner: {valor_retencion:.0f} €')
print(f'Coste de un contacto:        {COSTE_CONTACTO} €')
print()

filas = []
for u in [0.01, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.50]:
    yp  = (prob_mejor >= u).astype(int)
    tp  = int(((y_test==1)&(yp==1)).sum())
    fp  = int(((y_test==0)&(yp==1)).sum())
    fn  = int(((y_test==1)&(yp==0)).sum())
    al  = tp + fp
    rec = tp / (tp+fn)   if (tp+fn) > 0 else 0
    pre = tp / al         if al > 0      else 0
    f1  = 2*pre*rec/(pre+rec) if (pre+rec) > 0 else 0
    ben = tp * valor_retencion - al * COSTE_CONTACTO
    filas.append({'Umbral':u, 'Recall':rec, 'Precisión':pre, 'F1':f1,
                  'TP':tp, 'FP':fp, 'FN':fn, 'Alertas':al, 'Beneficio_EUR':int(ben)})

df_u = pd.DataFrame(filas)
U_OPT = df_u.loc[df_u['Beneficio_EUR'].idxmax(), 'Umbral']

print(f'{"Umbral":>8}  {"Recall":>7}  {"Precisión":>10}  {"F1":>6}  {"Alertas":>8}  {"Beneficio":>12}')
print('-' * 60)
for _, r in df_u.iterrows():
    marca = ' ← ÓPTIMO' if r['Umbral'] == U_OPT else ''
    print(f'{r["Umbral"]:>8.2f}  {r["Recall"]:>7.1%}  {r["Precisión"]:>10.1%}  '
          f'{r["F1"]:>6.3f}  {int(r["Alertas"]):>8,}  {int(r["Beneficio_EUR"]):>11,}€{marca}')
print(f'\nUmbral óptimo de negocio: {U_OPT}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Izquierda: recall, precisión y F1 por umbral
ax = axes[0]
ax.plot(df_u['Umbral'], df_u['Recall'],    color=C_CHURN,  marker='o', lw=2, label='Recall')
ax.plot(df_u['Umbral'], df_u['Precisión'], color=C_SAFE,   marker='s', lw=2, label='Precisión')
ax.plot(df_u['Umbral'], df_u['F1'],        color=C_BEST,   marker='^', lw=2, label='F1')
ax.axvline(U_OPT, color='orange', linestyle='--', lw=1.5,
           label=f'Óptimo negocio ({U_OPT})')
ax.set_xlabel('Umbral')
ax.set_title('Recall, Precisión y F1 según el umbral', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Derecha: beneficio neto
ax2 = axes[1]
colores_b = [C_BEST if u == U_OPT else C_SAFE for u in df_u['Umbral']]
ax2.bar(df_u['Umbral'].astype(str), df_u['Beneficio_EUR']/1000,
        color=colores_b, edgecolor='white')
ax2.set_xlabel('Umbral')
ax2.set_ylabel('Beneficio neto (miles €)')
ax2.set_title('Beneficio neto de campaña por umbral', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('El umbral óptimo no es 0.5 — depende del coste real del negocio',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 7. Curva de Ganancia

> *"Si llamo al X% de mis clientes con mayor score, ¿a qué % del churn llego?"*

Es la métrica que mejor entiende el negocio. Muestra cuánto mejor es el modelo
respecto a elegir clientes aleatoriamente.


In [ ]:
gain = (
    pd.DataFrame({'y_real': y_test.values, 'y_prob': prob_mejor})
    .sort_values('y_prob', ascending=False).reset_index(drop=True)
)
n_tot = len(gain)
n_ch  = gain['y_real'].sum()
gain['acum']        = gain['y_real'].cumsum()
gain['pct_clientes'] = (gain.index + 1) / n_tot * 100
gain['pct_churn']    = gain['acum'] / n_ch * 100
gain['lift']         = gain['pct_churn'] / gain['pct_clientes']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Curva de ganancia
ax = axes[0]
ax.plot(gain['pct_clientes'], gain['pct_churn'],
        color=C_CHURN, lw=2.5, label=f'{mejor_nombre} (AUC={mejor_auc:.3f})')
ax.plot([0,100], [0,100], color='gray', lw=1.5, linestyle='--', label='Sin modelo')
ax.fill_between(gain['pct_clientes'], gain['pct_churn'], gain['pct_clientes'],
                alpha=0.08, color=C_CHURN)
for pct in [10, 20, 30]:
    idx  = int(pct/100*n_tot)
    pch  = gain.iloc[idx]['pct_churn']
    ax.annotate(f'{pch:.0f}% del churn\ncon {pct}% de clientes',
                xy=(pct, pch), xytext=(pct+7, pch-12),
                arrowprops=dict(arrowstyle='->', color='black', lw=1),
                fontsize=8, bbox=dict(boxstyle='round,pad=0.2', fc='lightyellow', alpha=0.9))
ax.set_xlabel('% clientes contactados (mayor score primero)')
ax.set_ylabel('% churn total capturado')
ax.set_title('Curva de Ganancia', fontweight='bold')
ax.legend(fontsize=9); ax.set_xlim(0,100); ax.set_ylim(0,100)
ax.grid(True, alpha=0.3)

# Curva de lift
ax2 = axes[1]
ax2.plot(gain['pct_clientes'], gain['lift'], color=C_SAFE, lw=2.5)
ax2.axhline(1, color='gray', lw=1.5, linestyle='--')
ax2.fill_between(gain['pct_clientes'], gain['lift'], 1,
                 where=gain['lift']>=1, alpha=0.1, color=C_SAFE)
for pct in [10, 20, 30]:
    idx = int(pct/100*n_tot)
    lv  = gain.iloc[idx]['lift']
    ax2.annotate(f'Lift = {lv:.1f}x', xy=(pct, lv), xytext=(pct+5, lv+0.5),
                 arrowprops=dict(arrowstyle='->', color='black', lw=1),
                 fontsize=9, bbox=dict(boxstyle='round,pad=0.2', fc='lightblue', alpha=0.8))
ax2.set_xlabel('% clientes contactados (mayor score primero)')
ax2.set_ylabel('Veces mejor que sin modelo (lift)')
ax2.set_title('Curva de Lift', fontweight='bold')
ax2.set_xlim(0, 100); ax2.grid(True, alpha=0.3)

plt.suptitle('Con el modelo llegamos al mismo churn contactando a muchos menos clientes',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('── Tabla de ganancia ──')
print(f'{"% Clientes":>12}  {"% Churn capturado":>20}  {"Lift":>8}')
print('-'*44)
for pct in [5,10,15,20,25,30,40,50]:
    idx = int(pct/100*n_tot)
    r   = gain.iloc[idx]
    print(f'{pct:>12}%  {r["pct_churn"]:>19.1f}%  {r["lift"]:>8.2f}x')


## 8. Análisis de Falsos Negativos

Los **falsos negativos** son los churners que el modelo no detecta. Son el
error más costoso: nos perdemos una retención. ¿Tienen algo en común?
Comparamos su perfil con el de los churners que sí detectamos.


In [ ]:
y_pred_opt = (prob_mejor >= U_OPT).astype(int)

test_a = test.copy()
test_a['y_prob'] = prob_mejor
test_a['y_pred'] = y_pred_opt
test_a['y_real'] = y_test.values

test_a['grupo'] = np.select(
    [
        (test_a['y_real']==1) & (test_a['y_pred']==1),
        (test_a['y_real']==1) & (test_a['y_pred']==0),
        (test_a['y_real']==0) & (test_a['y_pred']==1),
        (test_a['y_real']==0) & (test_a['y_pred']==0),
    ],
    ['VP — detectado', 'FN — no detectado', 'FP — falsa alarma', 'VN — correcto'],
)

print('Distribución de predicciones:')
print(test_a['grupo'].value_counts().to_string())

vp = (test_a['grupo']=='VP — detectado').sum()
fn = (test_a['grupo']=='FN — no detectado').sum()
fp = (test_a['grupo']=='FP — falsa alarma').sum()
print(f'\nRecall:    {vp/(vp+fn):.1%} de los churners son detectados')
print(f'Precisión: {vp/(vp+fp):.1%} de las alertas son churners reales')
print(f'Contactos necesarios: {vp+fp:,} de {len(test):,} clientes ({(vp+fp)/len(test):.1%})')


In [ ]:
# Comparativa de perfiles VP vs FN
vars_comp = ['antiguedad_meses','ingreso_estimado','impago_mes_lag1',
             'stress_mes_lag1','ratio_estres_lag1','racha_impagos_lag1',
             'dias_retraso_lag1','critica_pendiente_lag1']

churners = test_a[test_a['y_real']==1]
perfil   = churners.groupby('grupo')[vars_comp].mean().T.round(3)
print('Perfil medio por tipo de predicción (solo churners):')
print(perfil[['VP — detectado','FN — no detectado']].to_string())


In [ ]:
# Distribución de densidad para las variables más discriminantes
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
vars_plot = ['ratio_estres_lag1', 'racha_impagos_lag1', 'antiguedad_meses']

vp_df = test_a[test_a['grupo']=='VP — detectado']
fn_df = test_a[test_a['grupo']=='FN — no detectado']

for ax, var in zip(axes, vars_plot):
    sns.kdeplot(vp_df[var].dropna(), ax=ax, color=C_BEST, fill=True, alpha=0.35,
                label='VP (detectado)', linewidth=2)
    sns.kdeplot(fn_df[var].dropna(), ax=ax, color=C_CHURN, fill=True, alpha=0.35,
                label='FN (no detectado)', linewidth=2)
    ax.set_title(var, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('¿En qué se diferencian los churners detectados de los no detectados?',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Los FN suelen tener menos señales de alarma (menor ratio_estres, menos impagos).')
print('Son los churners silenciosos: se van por razones que no están en los datos.')
print('Este es el límite informativo del modelo — no hay datos que los anticipen.')


## 9. Validación visual: hipótesis de multicausalidad

`ratio_estres_lag1 = (días_retraso_pago + 1) / (calidad_red + 1)`

Esta variable combina estrés económico y estrés de red. La hipótesis es que
la **combinación** de ambos problemas separa mejor churners de no-churners
que cada componente por separado.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

churn_test    = test_a[test_a['y_real']==1]
no_churn_test = test_a[test_a['y_real']==0]

for ax, var, titulo in zip(
    axes,
    ['ratio_estres_lag1', 'dias_retraso_lag1', 'stress_mes_lag1'],
    ['ratio_estres\n(combinado)', 'días_retraso\n(solo económico)', 'stress_red\n(solo red)']
):
    sns.kdeplot(no_churn_test[var].dropna(), ax=ax,
                color=C_SAFE, fill=True, alpha=0.35, label='No churn', lw=2)
    sns.kdeplot(churn_test[var].dropna(), ax=ax,
                color=C_CHURN, fill=True, alpha=0.35, label='Churn', lw=2)
    med_ch  = churn_test[var].median()
    med_nch = no_churn_test[var].median()
    ax.axvline(med_ch,  color=C_CHURN, linestyle='--', alpha=0.7)
    ax.axvline(med_nch, color=C_SAFE,  linestyle='--', alpha=0.7)
    ax.set_title(titulo, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('El ratio combinado separa mejor churners de no-churners que cada variable sola',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Separación entre medianas:')
for var in ['ratio_estres_lag1','dias_retraso_lag1','stress_mes_lag1']:
    mc  = churn_test[var].median()
    mnc = no_churn_test[var].median()
    diff = (mc-mnc)/mnc*100 if mnc!=0 else float('inf')
    print(f'  {var:30s}  churn={mc:.3f}  no-churn={mnc:.3f}  diff={diff:+.1f}%')


## 10. Walk-Forward Validation — estabilidad temporal del modelo

El split 80/20 garantiza que no hay leakage. Pero ¿sigue funcionando bien
el modelo conforme pasa el tiempo? Si el comportamiento de los clientes
cambia, el AUC irá bajando en los folds más recientes.

La Walk-Forward Validation mide esto: entrenamos con los primeros N meses
y validamos en los siguientes, expandiendo progresivamente la ventana.


In [ ]:
panel['fecha_dt'] = panel['ym'].dt.to_timestamp()
meses_ord = sorted(panel['fecha_dt'].unique())
n_meses   = len(meses_ord)
print(f'Meses disponibles: {n_meses}')
print(f'  Desde {meses_ord[0].strftime("%b %Y")} hasta {meses_ord[-1].strftime("%b %Y")}')

corte_ini = int(n_meses * 0.70)   # empieza con el 70% de los meses en train
paso      = 3                      # avanza 3 meses por fold

filas_wf = []
for corte in range(corte_ini, n_meses - 1, paso):
    f_corte = meses_ord[corte]
    f_fin   = meses_ord[min(corte + paso, n_meses - 1)]

    tr_wf = panel[panel['fecha_dt'] <= f_corte]
    te_wf = panel[(panel['fecha_dt'] > f_corte) & (panel['fecha_dt'] <= f_fin)]

    if len(te_wf) < 200 or te_wf['churn'].sum() < 10:
        continue

    try:
        prep_wf = ColumnTransformer([
            ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), FEATURES_NUM),
            ('cat', Pipeline([
                ('imp', SimpleImputer(strategy='constant', fill_value='desconocido')),
                ('enc', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False)),
            ]), FEATURES_CAT),
        ])
        m_wf = Pipeline([
            ('prep', prep_wf),
            ('model', LGBMClassifier(
                n_estimators=300, learning_rate=0.05, max_depth=5,
                num_leaves=31, min_child_samples=50,
                scale_pos_weight=int((tr_wf['churn']==0).sum()/(tr_wf['churn']==1).sum()),
                random_state=RANDOM_STATE, verbose=-1
            ))
        ])
        m_wf.fit(tr_wf[FEATURES_NUM + FEATURES_CAT], tr_wf['churn'])
        prob_wf = m_wf.predict_proba(te_wf[FEATURES_NUM + FEATURES_CAT])[:, 1]
        auc_wf  = roc_auc_score(te_wf['churn'], prob_wf)
        filas_wf.append({
            'Train hasta': f_corte.strftime('%b %Y'),
            'Test':        f'{meses_ord[corte+1].strftime("%b %Y")} - {f_fin.strftime("%b %Y")}',
            'N train': len(tr_wf), 'N test': len(te_wf),
            'Tasa churn test': round(te_wf['churn'].mean(), 4),
            'AUC': round(auc_wf, 4),
        })
    except Exception as e:
        print(f'  Fold {corte}: {e}')

if filas_wf:
    df_wf = pd.DataFrame(filas_wf)
    print()
    print(df_wf.to_string(index=False))
    print(f'\nAUC medio Walk-Forward: {df_wf["AUC"].mean():.4f}')
    print(f'AUC std Walk-Forward:   {df_wf["AUC"].std():.4f}')
    print(f'AUC split 80/20:        {mejor_auc:.4f}')
    print('\nSi el AUC es estable a lo largo del tiempo: el modelo no sufre deriva.')
    print('Si baja en los folds más recientes: el modelo necesitaría reentrenarse.')
else:
    print('Datos insuficientes para Walk-Forward en este dataset.')
    df_wf = pd.DataFrame()


In [ ]:
if not df_wf.empty and len(df_wf) >= 2:
    fig, ax = plt.subplots(figsize=(10, 4))
    x = range(len(df_wf))
    ax.plot(x, df_wf['AUC'], marker='o', color=C_SAFE, lw=2.5, markersize=8,
            label='AUC por fold temporal')
    ax.axhline(mejor_auc, color=C_CHURN, lw=2, linestyle='--',
               label=f'AUC split 80/20 = {mejor_auc:.4f}')
    ax.axhline(df_wf['AUC'].mean(), color=C_BEST, lw=1.5, linestyle=':',
               label=f'Media WF = {df_wf["AUC"].mean():.4f}')
    ax.fill_between(x,
                    df_wf['AUC'].mean() - df_wf['AUC'].std(),
                    df_wf['AUC'].mean() + df_wf['AUC'].std(),
                    alpha=0.12, color=C_SAFE, label='±1 std')
    ax.set_xticks(x)
    ax.set_xticklabels(df_wf['Train hasta'], rotation=30, fontsize=9)
    ax.set_xlabel('Fecha de corte del entrenamiento')
    ax.set_ylabel('AUC en test temporal')
    ax.set_title('Walk-Forward Validation — ¿Se degrada el modelo con el tiempo?',
                 fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.55, 0.80)
    plt.tight_layout()
    plt.show()


## 11. Resumen completo de todas las iteraciones


In [ ]:
tabla = pd.DataFrame([
    {'Iteración': 'Binario LR',         'AUC': 0.991, 'Leakage': 'Sí', 'Producción': 'No',
     'Comentario': 'AUC inflado: n_meses_facturados = proxy del target'},
    {'Iteración': 'Binario RF',         'AUC': 0.994, 'Leakage': 'Sí', 'Producción': 'No',
     'Comentario': 'Mismo problema que LR binario'},
    {'Iteración': 'Temporal LR base',   'AUC': 0.685, 'Leakage': 'No', 'Producción': 'Sí',
     'Comentario': 'Primer modelo honesto sin leakage'},
    {'Iteración': 'LR + GridSearch',    'AUC': 0.690, 'Leakage': 'No', 'Producción': 'Sí',
     'Comentario': 'C=0.01, L1, class_weight=balanced'},
    {'Iteración': 'LR + Tendencia',     'AUC': 0.701, 'Leakage': 'No', 'Producción': 'Sí',
     'Comentario': 'racha_impagos, sin_consumo_2m, rolling 3m'},
    {'Iteración': 'LR + Features V2',   'AUC': 0.703, 'Leakage': 'No', 'Producción': 'Sí',
     'Comentario': 'ratio_estres, critica_pendiente, sentimiento_lag1'},
    {'Iteración': 'LightGBM V3',        'AUC': mejor_auc if mejor_nombre=='LightGBM' else roc_auc_score(y_test,prob_lgbm),
     'Leakage': 'No', 'Producción': 'Sí',
     'Comentario': 'Gradient Boosting, scale_pos_weight, 500 estimadores'},
    {'Iteración': 'XGBoost V3',         'AUC': roc_auc_score(y_test,prob_xgb),
     'Leakage': 'No', 'Producción': 'Sí',
     'Comentario': 'Gradient Boosting, max_depth=4, subsample=0.8'},
])

print(tabla.to_string(index=False))
print(f'\nMejor modelo global: {mejor_nombre} — AUC = {mejor_auc:.4f}')


## 12. Conclusiones

### ¿El Walk-Forward mejora o empeora?
El Walk-Forward no mejora el AUC — ese no es su objetivo. Mide si el modelo
es **estable a lo largo del tiempo**. Si el AUC de los folds recientes es
similar al de los primeros, el modelo no sufre deriva y se puede desplegar
con confianza. Si baja, necesitaría reentrenamiento periódico.

### ¿La V3 mejora a la V2?
El Gradient Boosting (LightGBM/XGBoost) probablemente supera al AUC = 0.703
de la LR. Si la mejora es modesta (~0.01), la LR sigue siendo preferible
en producción por su **interpretabilidad**: los coeficientes son directamente
explicables al equipo comercial. Si el GBM supera en ~0.02 o más, vale la
pena el coste de interpretabilidad adicional que cubre SHAP.

### Tres convergen, lo que confirma el techo informativo
Que LR, LightGBM y XGBoost den resultados similares en el rango 0.70-0.74
indica que estamos cerca del **límite informativo de los datos disponibles**.
El churn tiene una componente aleatoria (oferta de la competencia, cambio
personal) que ningún modelo puede capturar con los datos actuales.

### Próximos pasos reales (para que la empresa los implemente)
1. **Aplicación comercial con avisos diarios**: cada mañana, el modelo
   genera la lista de clientes con mayor riesgo. Los comerciales ven en
   su app quién llamar hoy, registran el contacto, el resultado y las
   observaciones. El historial retroalimenta el modelo.
2. **Reentrenamiento mensual automatizado**: pipeline en producción que
   ejecuta el modelo con datos nuevos cada mes y alerta si el AUC cae.
3. **Segmentación de la oferta de retención**: no ofrecer lo mismo a un
   cliente Prepago rural (descuento económico) que a un Premium urbano
   (mejora de servicio). El perfil del churner ya está identificado.
4. **Modelo de supervivencia**: en lugar de predecir si/no, predecir
   *cuándo* se irá el cliente. Permite priorizar mejor la urgencia.
5. **Dashboard operativo** (ya construido en este proyecto): conectar
   las predicciones mensuales al dashboard para que el equipo comercial
   tenga visibilidad completa sin necesidad de acceder a Python.

---
*Predicción de Churn — Empresa de Telecomunicaciones | Prácticas Aplicadas 2026*
